# Assignment 3, Person 1: main runs of the proposed system (Kaggle)

From a fresh **git clone** (so every `run_info.json` records the commit):

| Cell | Run | Time |
|---|---|---|
| 6 | **E2**: frozen XLM-R at the **last layer** (`frozen_last`), no training | ~4 min |
| 7 | **E1**: proposed system, **seed 87** | ~36-45 min |
| 8 | **E1**: proposed system, **seed 1234** | ~36-45 min |
| 9 | **E1**: proposed system, **seed 2027** | ~36-45 min |

**Settings (right-hand panel > Session options):** Accelerator **GPU T4 x2** (not P100), Internet **On**.
Input: your private dataset with `sentences.bin` (= `sentences.zip` renamed) (Add Input). Run with **Save Version > Save & Run All (Commit)**:
it runs in the background and keeps everything in `/kaggle/working` as the notebook's Output.

In [ ]:
# Cell 1: find the synthetic English set in /kaggle/input
import os, glob, subprocess
OUT  = '/kaggle/working'
RUNS = f'{OUT}/runs_a3'            # run folders (with vectors/ for Person 2)
LOGS = f'{OUT}/gpu_logs'
CKPT = '/tmp/kg/checkpoints'   # Stage A checkpoints (~1.1 GB per seed, not kept in the output)
for d in (RUNS, LOGS, CKPT): os.makedirs(d, exist_ok=True)

# The dataset holds sentences.bin (= sentences.zip renamed, so Kaggle does not try to unpack it:
# Kaggle rejects 4 file names with an apostrophe). We unpack it here; Linux accepts those names.
found = [p for p in glob.glob('/kaggle/input/**/generated_sentences', recursive=True) if os.path.isdir(p)]
if not found:
    archives = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
                if os.path.isfile(p) and os.path.basename(p).startswith('sentences')]
    assert archives, 'sentences.bin not found: Add Input > your dataset'
    print('unpacking', archives[0], '(about 1 min)')
    !mkdir -p /tmp/kg/synthetic && unzip -q -o "{archives[0]}" -d /tmp/kg/synthetic
    found = [p for p in glob.glob('/tmp/kg/synthetic/**/generated_sentences', recursive=True) if os.path.isdir(p)]
GEN = found[0]
print('synthetic set:', GEN, '|', len(glob.glob(f'{GEN}/*/*.jsonl')), 'word files (about 2,345 expected)')

In [ ]:
# Cell 2: fresh git clone of the team repo, packages, GPU check
%cd /kaggle/working
!rm -rf /tmp/kg/SenseShift
!git clone -q https://github.com/abiha25/SenseShift-Semantic-change-across-multiple-time-periods.git /tmp/kg/SenseShift
%cd /tmp/kg/SenseShift
!git log --oneline -1
!pip install -q bcubed nltk pyyaml
import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version('4.46'):
    !pip install -q -U "transformers>=4.46"
    print('transformers upgraded: restart is NOT needed for the !python runs below')
import torch
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Accelerator to GPU T4 x2')

Link the synthetic set into the clone (`data/external/` is ignored by git, so the clone stays clean).
`git status` must print **nothing**: then the recorded commit is exactly the code that ran.

In [ ]:
# Cell 3: link the synthetic set, check the clone is clean
!mkdir -p data/external/synthetic_en
!ln -sfn "{GEN}" data/external/synthetic_en/generated_sentences
!echo "word files via the link:" $(ls data/external/synthetic_en/generated_sentences/*/*.jsonl | wc -l)
!git status --porcelain

Expected last line: `41 passed`.

In [ ]:
# Cell 4: tests
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -1

In [ ]:
# Cell 5: GPU memory logger (for the compute cost in Section 6)
def start_gpu_log(name):
    f = open(f'{LOGS}/{name}.csv', 'w')
    return subprocess.Popen(['nvidia-smi', '--id=0', '--query-gpu=timestamp,memory.used',
                             '--format=csv,noheader,nounits', '-l', '5'], stdout=f, stderr=subprocess.DEVNULL)
def peak_gpu_mib(name):
    vals = [int(l.split(',')[1]) for l in open(f'{LOGS}/{name}.csv') if ',' in l]
    return max(vals) if vals else None
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # the code uses one GPU; keep runs on GPU 0

## E2: frozen encoder at the last layer
Same pipeline as the proposed system, but the encoder is frozen XLM-R at the **last layer** (the fine-tuned encoder also
uses the last layer). Own name `frozen_last`, so it does not replace the Assignment 2 layer-8 run. No randomness: one run.

In [ ]:
# Cell 6: E2 (~4 min)
log = start_gpu_log('frozen_last')
!python -m src.proposed run --name frozen_last --set training.enabled=false --set model.layer=-1 \
    --set output.results_dir="{RUNS}" 2>&1 | grep --line-buffered -vE "^\s+[a-zäöå]+\s+[0-9]"
log.terminate(); print('peak GPU memory (MiB):', peak_gpu_mib('frozen_last'))

## E1: main system, seeds 87, 1234, 2027
Exactly the Assignment 2 configuration; only the seed changes. Each seed trains its own Stage A, then the 10 folds.

In [ ]:
# Cell 7: seed 87 (~36-45 min)
log = start_gpu_log('proposed_seed87')
!python -m src.proposed run --set seed=87 \
    --set output.checkpoints_dir="{CKPT}" --set output.results_dir="{RUNS}" 2>&1 | grep --line-buffered -vE "^\s+[a-zäöå]+\s+[0-9]"
log.terminate(); print('peak GPU memory (MiB):', peak_gpu_mib('proposed_seed87'))
!rm -rf {CKPT}/*      # free disk space; the run folder keeps everything needed

In [ ]:
# Cell 8: seed 1234 (~36-45 min)
log = start_gpu_log('proposed_seed1234')
!python -m src.proposed run --set seed=1234 \
    --set output.checkpoints_dir="{CKPT}" --set output.results_dir="{RUNS}" 2>&1 | grep --line-buffered -vE "^\s+[a-zäöå]+\s+[0-9]"
log.terminate(); print('peak GPU memory (MiB):', peak_gpu_mib('proposed_seed1234'))
!rm -rf {CKPT}/*      # free disk space; the run folder keeps everything needed

In [ ]:
# Cell 9: seed 2027 (~36-45 min)
log = start_gpu_log('proposed_seed2027')
!python -m src.proposed run --set seed=2027 \
    --set output.checkpoints_dir="{CKPT}" --set output.results_dir="{RUNS}" 2>&1 | grep --line-buffered -vE "^\s+[a-zäöå]+\s+[0-9]"
log.terminate(); print('peak GPU memory (MiB):', peak_gpu_mib('proposed_seed2027'))
!rm -rf {CKPT}/*      # free disk space; the run folder keeps everything needed

## Check the runs
Every run should show a commit, `uncommitted = False`, and its scores.

In [ ]:
# Cell 10: summary of the new runs
import json
import pandas as pd
rows = []
for d in sorted(glob.glob(f'{RUNS}/*')):
    if not os.path.exists(f'{d}/scores.json'):
        print('UNFINISHED (no scores.json):', os.path.basename(d)); continue
    info = json.load(open(f'{d}/run_info.json')); s = json.load(open(f'{d}/scores.json'))
    sel = json.load(open(f'{d}/selection.json'))
    sh = s.get('subtask1', {}).get('subtask1b', {}).get('shuffle_control') or {}
    git = info.get('git') or {}
    rows.append({'run': os.path.basename(d), 'commit': (git.get('commit') or 'NONE')[:7],
                 'uncommitted': git.get('uncommitted_changes'), 'gpu': info['environment'].get('gpu'),
                 '1a': s['scores'].get('1a_bcubed_f1'), '1b': s['scores'].get('1b_spearman'),
                 '1b shuffle': sh.get('shuffled_mean'), '2': s['scores'].get('2_macro_f1'),
                 'APD': s['scores'].get('graded_spearman'),
                 'minutes': round(sel.get('timing', {}).get('total_seconds', float('nan')) / 60, 1),
                 'vector files': len(glob.glob(f'{d}/vectors/*.npz'))})
table = pd.DataFrame(rows).round(3)
table.to_csv(f'{OUT}/a3_main_runs_summary.csv', index=False)
table

In [ ]:
# Cell 11: one small zip without the vectors (for the repo); the full folders with vectors stay in the Output
!cd /kaggle/working && rm -f a3_main_runs.zip && zip -qr a3_main_runs.zip runs_a3 gpu_logs a3_main_runs_summary.csv -x "*/vectors/*"
!ls -la /kaggle/working